# 01 — Curate ChEMBL extract

Phase 1 of **CRC_Inhibitor_ML**. Takes the raw ChEMBL pull from Phase 0 and produces a clean, ML-ready labeled dataset. Cleanup pipeline:

1. **Quality filter** — drop censored measurements, low-confidence assays, off-target types
2. **Standardize SMILES** with RDKit — canonicalize, strip salts, neutralize charges
3. **Harmonize pIC50** — prefer `pchembl_value`; compute from `standard_value + standard_units` for rows where pchembl is missing
4. **Deduplicate** — one row per (target, molecule); take median pIC50 across replicate measurements
5. **EDA** on the clean data — counts per target, pIC50 distributions
6. **Save** to `data/interim/chembl_crc_targets_clean.csv` for Phase 2 (featurization + baseline)

Input:  `data/raw/chembl_crc_targets_raw.csv` (41,343 rows from Phase 0)
Output: `data/interim/chembl_crc_targets_clean.csv`

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from rdkit import Chem, RDLogger
from rdkit.Chem.MolStandardize import rdMolStandardize

# Silence RDKit's noisy per-molecule warnings (we keep failures as NaN and drop them)
RDLogger.DisableLog("rdApp.*")
tqdm.pandas()

PROJECT_ROOT     = Path(r"C:\Users\palla\OneDrive\Documents\Coding Projects\CRC_Inhibitor_ML")
RAW_CSV          = PROJECT_ROOT / "data" / "raw"     / "chembl_crc_targets_raw.csv"
INTERIM_DATA_DIR = PROJECT_ROOT / "data" / "interim"
INTERIM_DATA_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_CSV        = INTERIM_DATA_DIR / "chembl_crc_targets_clean.csv"

print(f"Input:  {RAW_CSV}")
print(f"Output: {CLEAN_CSV}")

## Cell 1 — load raw extract

Read what Phase 0 saved. Verify the row count matches what we expected (~41K).

In [ ]:
raw_df = pd.read_csv(RAW_CSV)
print(f"Loaded {len(raw_df):,} rows, {raw_df.shape[1]} columns")
print(f"Columns: {list(raw_df.columns)}")
raw_df.head(3)

## Cell 2 — quality filter

Three sequential cuts. Each filter shows running counts so we know what each step costs.

| Filter | Why |
|---|---|
| `standard_relation == '='` | Drop censored values (`>10 µM` style) that pretend to be exact measurements. Phase 0 EDA confirmed the CRC targets are clean here but other targets won't be. |
| `confidence_score >= 8` | ChEMBL's confidence that the assay actually hit *our* target (vs. a downstream cellular readout). 8-9 = high confidence the assay is target-specific. |
| `assay_type in ('B', 'F')` | Binding (B) or functional (F) — both meaningful for potency. Excludes A (ADMET), T (toxicity), P (physicochemical) — those measure other things. |

In [ ]:
def show(label, n):
    print(f"{label:<40} {n:>10,}")

show("Raw rows:", len(raw_df))

filtered = raw_df[raw_df["standard_relation"] == "="].copy()
show("After relation == '=':", len(filtered))

filtered = filtered[filtered["confidence_score"] >= 8]
show("After confidence_score >= 8:", len(filtered))

filtered = filtered[filtered["assay_type"].isin(["B", "F"])]
show("After assay_type in (B, F):", len(filtered))

print()
print("Per-target survival:")
print(filtered["target_name"].value_counts())

## Cell 3 — standardize SMILES

RDKit pipeline on each SMILES string:

1. **Parse** the SMILES → RDKit `Mol`. If parse fails, return None (we'll drop these rows).
2. **LargestFragmentChooser** — many ChEMBL entries are salt forms like `[Cl-].CC(=O)[NH3+]...`. Take the largest fragment — that's the actual drug molecule, the rest is counter-ion / solvent.
3. **Uncharger** — neutralize protonation states where possible. Bioactivity is reported on the neutral molecule, but ChEMBL stores whatever protonation was in the source paper.
4. **Canonical SMILES** — RDKit's canonical form. Critical: ensures `"CCO"` and `"OCC"` (both ethanol) become the same string, so dedup works correctly.

Takes ~30–60 seconds for our row count (tqdm progress bar shows status).

In [ ]:
_largest   = rdMolStandardize.LargestFragmentChooser()
_uncharger = rdMolStandardize.Uncharger()

def standardize_smiles(smi):
    if pd.isna(smi):
        return None
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    try:
        mol = _largest.choose(mol)
        mol = _uncharger.uncharge(mol)
        return Chem.MolToSmiles(mol, canonical=True)
    except Exception:
        return None

filtered["canonical_smiles_std"] = filtered["canonical_smiles"].progress_apply(standardize_smiles)

n_failed = filtered["canonical_smiles_std"].isna().sum()
filtered = filtered.dropna(subset=["canonical_smiles_std"]).copy()
print(f"Standardization failed on {n_failed:,} rows (dropped). Remaining: {len(filtered):,}")

## Cell 4 — harmonize pIC50 labels

We need one numerical label per row. ChEMBL's `pchembl_value` is the pre-computed pIC50 and is preferred when available. When it's NaN, we compute it ourselves:

$$\text{pIC50} = -\log_{10}(\text{IC50 in M})$$

where the IC50 in M is `standard_value × unit_multiplier`. We support the common units (nM, µM, mM, M, pM). Anything else gets NaN and gets dropped.

Vectorized to keep the operation fast on the full filtered set.

In [ ]:
UNIT_TO_M = {
    "nM": 1e-9,
    "uM": 1e-6,
    "\u03bcM": 1e-6,   # the unicode mu version
    "mM": 1e-3,
    "M":  1.0,
    "pM": 1e-12,
}

# Vectorized: compute pIC50 for every row from value + units, then prefer pchembl where present
multipliers = filtered["standard_units"].map(UNIT_TO_M)
molar       = filtered["standard_value"] * multipliers

with np.errstate(invalid="ignore", divide="ignore"):
    computed_pic50 = -np.log10(molar)
computed_pic50 = computed_pic50.where(molar > 0)  # negatives/zeros → NaN

filtered["pic50"] = filtered["pchembl_value"].fillna(computed_pic50)

n_no_label = filtered["pic50"].isna().sum()
filtered = filtered.dropna(subset=["pic50"]).copy()
print(f"Rows with no usable pIC50 label: {n_no_label:,} (dropped). Remaining: {len(filtered):,}")

print(f"\npIC50 source breakdown:")
from_pchembl = filtered["pchembl_value"].notna().sum()
from_compute = len(filtered) - from_pchembl
print(f"  From pchembl_value:       {from_pchembl:>7,}")
print(f"  Computed from value+units: {from_compute:>7,}")

## Cell 5 — deduplicate by (target, molecule)

Many molecules appear multiple times in ChEMBL — same compound, different papers, slightly different IC50 values. For training we want **one row per (target, molecule)** pair.

Aggregation strategy:
- `pic50` = **median** of all replicate measurements (robust to outliers, unlike mean)
- `pic50_std` = standard deviation across replicates (tells us measurement variability)
- `n_measurements` = how many replicate measurements were combined

Why median over mean: experimental IC50 measurements have heavy tails (occasional outlier values from edge-case assays). Median ignores them.

In [ ]:
clean = (
    filtered
    .groupby(
        ["target_chembl_id", "target_name", "canonical_smiles_std"],
        as_index=False,
    )
    .agg(
        pic50=("pic50", "median"),
        pic50_std=("pic50", "std"),
        n_measurements=("pic50", "size"),
    )
)

print(f"Rows before dedup: {len(filtered):,}")
print(f"Rows after dedup:  {len(clean):,}  (unique target+molecule pairs)")
print()
print("Per-target final counts:")
print(clean["target_name"].value_counts())
print()
print("Replicate stats (how many measurements per unique target+molecule):")
print(clean["n_measurements"].describe())

## Cell 6 — EDA on the clean data

Same per-target pIC50 histograms as Phase 0, but now on the curated set. Expectation: the overall shapes look similar to the raw data (the underlying biology hasn't changed), but the spikes should be slightly smoothed because dedup averaged replicate measurements that previously landed in the same bin.

In [ ]:
# Filter by ChEMBL ID (exact) and use short names for titles
TARGET_DISPLAY = {
    "CHEMBL2189121": "KRAS",
    "CHEMBL5145":    "BRAF",
    "CHEMBL203":     "EGFR",
    "CHEMBL4005":    "PIK3CA",
}

fig, axes = plt.subplots(2, 2, figsize=(11, 6), sharex=True)
for ax, (cid, name) in zip(axes.flat, TARGET_DISPLAY.items()):
    subset = clean.loc[clean["target_chembl_id"] == cid, "pic50"]
    ax.hist(subset, bins=40)
    ax.set_title(f"{name}  (n={len(subset):,} unique molecules)")
    ax.set_xlabel("pIC50")
    ax.set_ylabel("count")
plt.tight_layout()
plt.show()

## Cell 7 — save curated CSV

Output schema for Phase 2:

| Column | Meaning |
|---|---|
| `target_chembl_id` | One of the four CRC targets |
| `target_name` | Human-readable target name |
| `canonical_smiles_std` | Cleaned, canonicalized SMILES — the ML input |
| `pic50` | Median pIC50 across replicate measurements — the ML label |
| `pic50_std` | Std dev of replicates (measurement noise estimate) |
| `n_measurements` | How many raw rows collapsed into this dedup entry |

In [ ]:
clean.to_csv(CLEAN_CSV, index=False)
print(f"Saved {len(clean):,} rows to {CLEAN_CSV}")
print(f"\nFinal column types:")
print(clean.dtypes)